# 🌍 Global Warming and Ocean Impact Analysis

Exploratory Data Analysis (EDA) on global land temperature trends (1750-2015) and their relationship with ocean temperature, using the Berkeley Earth Surface Temperature dataset.

This notebook covers:
- Loading and cleaning historical temperature data
- Yearly and decadal temperature trends
- Which countries have warmed the most
- Seasonal temperature patterns
- Comparison of land-only vs land+ocean temperature trends

**Full project (with reusable Python modules, automated data download, and Git workflow):**
[GitHub - Global-Warming-And-Ocean-Impact-Analysis](https://github.com/anshikaraj-hub/Global-Warming-And-Ocean-Impact-Analysis)


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import calendar
import os

In [2]:
import plotly.io as pio
pio.renderers.default = 'iframe'

## 1. Loading the Data

The dataset contains two files we'll use:
- `GlobalTemperatures.csv` — monthly global average land temperature since 1750
- `GlobalLandTemperaturesByCountry.csv` — monthly temperature for every country since 1743

First, let's confirm the file paths Kaggle has mounted for this dataset.

In [3]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data/GlobalTemperatures.csv
/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data/GlobalLandTemperaturesByState.csv
/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data/GlobalLandTemperaturesByCountry.csv
/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data/GlobalLandTemperaturesByCity.csv
/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data/GlobalLandTemperaturesByMajorCity.csv


In [4]:
# Update this path if it differs from the listing above
DATA_DIR = '/kaggle/input/datasets/organizations/berkeleyearth/climate-change-earth-surface-temperature-data'

global_temps = pd.read_csv(os.path.join(DATA_DIR, 'GlobalTemperatures.csv'))
country_temps = pd.read_csv(os.path.join(DATA_DIR, 'GlobalLandTemperaturesByCountry.csv'))

print(f"GlobalTemperatures: {global_temps.shape}")
print(f"GlobalLandTemperaturesByCountry: {country_temps.shape}")

global_temps.head()

GlobalTemperatures: (3192, 9)
GlobalLandTemperaturesByCountry: (577462, 4)


,dt,LandAverageTemperature,LandAverageTemperatureUncertainty,LandMaxTemperature,LandMaxTemperatureUncertainty,LandMinTemperature,LandMinTemperatureUncertainty,LandAndOceanAverageTemperature,LandAndOceanAverageTemperatureUncertainty
0,1750-01-01,3.034,3.574,NaN,NaN,NaN,NaN,NaN,NaN
1,1750-02-01,3.083,3.702,NaN,NaN,NaN,NaN,NaN,NaN
2,1750-03-01,5.626,3.076,NaN,NaN,NaN,NaN,NaN,NaN
3,1750-04-01,8.490,2.451,NaN,NaN,NaN,NaN,NaN,NaN
4,1750-05-01,11.573,2.072,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Data Cleaning

Before analysis, we:
- Convert the `dt` column to a proper datetime type (it loads as a string)
- Keep only the columns relevant to this analysis
- Forward-fill small gaps in `LandAverageTemperature` and country `AverageTemperature`

**Note:** `LandAndOceanAverageTemperature` has missing values before 1850. This isn't a data quality issue to "fix" — ocean surface temperature measurement technology simply didn't exist before then. We deliberately leave these as `NaN` rather than fabricating values.

In [5]:
def clean_data(df, dataset_type):
    df = df.copy()
    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')

    if dataset_type == "global":
        df = df[['dt', 'LandAverageTemperature', 'LandAndOceanAverageTemperature']]
        df['LandAverageTemperature'] = df['LandAverageTemperature'].ffill()

    elif dataset_type == "country":
        df = df[['dt', 'AverageTemperature', 'Country']]
        df['AverageTemperature'] = df['AverageTemperature'].ffill()

    print(f"Cleaned {dataset_type} dataset: {df.shape}")
    print(f"Missing values:\n{df.isnull().sum()}\n")
    return df

clean_global = clean_data(global_temps, "global")
clean_country = clean_data(country_temps, "country")

Cleaned global dataset: (3192, 3)
Missing values:
dt                                   0
LandAverageTemperature               0
LandAndOceanAverageTemperature    1200
dtype: int64

Cleaned country dataset: (577462, 3)
Missing values:
dt                    0
AverageTemperature    0
Country               0
dtype: int64



## 3. Exploratory Data Analysis

### 3.1 Yearly Average Temperatures

Monthly data is aggregated into yearly averages to see the long-term trend more clearly.

In [6]:
def calculate_yearly_averages(df, column_name):
    if column_name == "land":
        return df.groupby(df['dt'].dt.year)['LandAverageTemperature'].mean().reset_index()
    elif column_name == "land_and_ocean":
        return df.groupby(df['dt'].dt.year)['LandAndOceanAverageTemperature'].mean().reset_index()
    elif column_name == "country":
        return df.groupby(df['dt'].dt.year)['AverageTemperature'].mean().reset_index()

yearly_land = calculate_yearly_averages(clean_global, "land")
yearly_land_and_ocean = calculate_yearly_averages(clean_global, "land_and_ocean")

print(f"Data range: {yearly_land['dt'].min()} - {yearly_land['dt'].max()}")
yearly_land.head()

Data range: 1750 - 2015


,dt,LandAverageTemperature
0,1750,8.523333
1,1751,9.112417
2,1752,6.362667
3,1753,8.388083
4,1754,8.469333


### 3.2 Decadal Analysis — Sharpest Temperature Increases

Grouping by decade and looking at the change (`.diff()`) between consecutive decades shows *where* warming accelerated — not just which decade was hottest overall.

In [7]:
def decadal_analysis(df, column_name):
    df = df.copy()
    df['decade'] = (df['dt'].dt.year // 10) * 10

    col_map = {
        "land": "LandAverageTemperature",
        "land_and_ocean": "LandAndOceanAverageTemperature",
        "country": "AverageTemperature"
    }
    col = col_map[column_name]

    decade_avg = df.groupby('decade')[col].mean().reset_index()
    decade_avg['temp_increase'] = decade_avg[col].diff()
    sharpest = decade_avg.loc[decade_avg['temp_increase'].idxmax()]

    return decade_avg, sharpest

decade_land, sharpest_land = decadal_analysis(clean_global, "land")
decade_land_and_ocean, sharpest_land_ocean = decadal_analysis(clean_global, "land_and_ocean")

print("Sharpest decadal increase (Land only):")
print(sharpest_land)
print("\nSharpest decadal increase (Land + Ocean):")
print(sharpest_land_ocean)

Sharpest decadal increase (Land only):
decade                    1820.000000
LandAverageTemperature       8.182233
temp_increase                0.931167
Name: 7, dtype: float64

Sharpest decadal increase (Land + Ocean):
decade                            2000.000000
LandAndOceanAverageTemperature      15.785967
temp_increase                        0.187525
Name: 25, dtype: float64


### 3.3 Which Countries Have Warmed the Most?

**Methodology note:** a naive approach (comparing each country's first vs. last *single* recorded temperature) is misleading — seasonal differences alone can produce huge false "warming" values (e.g. comparing a January reading to a July reading). Instead, we compare the **average of each country's first 10 years** against the **average of its last 10 years**, which removes seasonal bias.

In [8]:
def top_warming_countries(df):
    results = {}
    for country, group in df.groupby('Country'):
        first_year = group['dt'].dt.year.min()
        last_year = group['dt'].dt.year.max()

        first_decade_avg = group[group['dt'].dt.year <= first_year + 10]['AverageTemperature'].mean()
        last_decade_avg = group[group['dt'].dt.year >= last_year - 10]['AverageTemperature'].mean()

        results[country] = last_decade_avg - first_decade_avg

    warming = pd.Series(results)
    return warming.sort_values(ascending=False).head(10)

top_countries = top_warming_countries(clean_country)
print("Top 10 countries by temperature increase (first decade vs. last decade average):")
top_countries

Top 10 countries by temperature increase (first decade vs. last decade average):


Uzbekistan              13.907154
Turkmenistan            12.222845
China                    6.859022
Egypt                    6.758399
United Arab Emirates     6.391825
Djibouti                 5.937618
Bahamas                  5.265297
Eritrea                  5.071887
Oman                     4.904734
Åland                    4.604942
dtype: float64

### 3.4 Seasonal Temperature Patterns

Aggregating by calendar month (regardless of year) reveals the global seasonal cycle.

In [9]:
def seasonal_analysis(df):
    monthly_avg = df.groupby(df['dt'].dt.month)['LandAverageTemperature'].mean().reset_index()

    hottest = monthly_avg.loc[monthly_avg['LandAverageTemperature'].idxmax()]
    coldest = monthly_avg.loc[monthly_avg['LandAverageTemperature'].idxmin()]

    print(f"Hottest month: {calendar.month_name[int(hottest['dt'])]} ({hottest['LandAverageTemperature']:.2f}°C)")
    print(f"Coldest month: {calendar.month_name[int(coldest['dt'])]} ({coldest['LandAverageTemperature']:.2f}°C)")

    return monthly_avg

monthly_avg = seasonal_analysis(clean_global)

Hottest month: July (14.28°C)
Coldest month: January (2.28°C)


## 4. Visualizations

Interactive charts built with Plotly — hover for exact values, drag to zoom, click legend items to toggle lines on/off.

### 4.1 Global Land Temperature Trend (1750-2015)

In [10]:
fig = px.line(yearly_land, x='dt', y='LandAverageTemperature',
               title='Global Land Temperature Trend (1750-2015)',
               labels={'dt': 'Year', 'LandAverageTemperature': 'Average Temperature (°C)'})
fig.show()

Notice the wild swings before ~1850 (less reliable measurements), the dip around 1810-1820 (linked to the 1815 Mount Tambora eruption causing global cooling), and the steady, sharp rise from ~1975 onward.

### 4.2 Top 10 Countries by Temperature Increase

In [11]:
top_countries_df = top_countries.reset_index()
top_countries_df.columns = ['Country', 'Warming']

fig = px.bar(top_countries_df, x='Country', y='Warming',
              title='Top 10 Countries by Temperature Increase',
              color='Warming', color_continuous_scale='Reds',
              labels={'Warming': 'Temperature Increase (°C)'})
fig.show()

Central Asian and arid/continental countries (Uzbekistan, Turkmenistan) show the sharpest long-term warming — consistent with how continental climates amplify temperature change compared to coastal/maritime regions.

### 4.3 Average Temperature by Decade

In [12]:
fig = px.bar(decade_land, x='decade', y='LandAverageTemperature',
              title='Average Land Temperature by Decade',
              color='LandAverageTemperature', color_continuous_scale='Oranges',
              labels={'LandAverageTemperature': 'Average Temperature (°C)'})
fig.show()

### 4.4 Seasonal Temperature Pattern

In [13]:
monthly_avg['Month_Name'] = monthly_avg['dt'].apply(lambda x: calendar.month_name[int(x)])

fig = px.line(monthly_avg, x='Month_Name', y='LandAverageTemperature',
               title='Seasonal Temperature Pattern (Global Average)',
               markers=True,
               labels={'LandAverageTemperature': 'Average Temperature (°C)', 'Month_Name': 'Month'})
fig.show()

Temperatures peak in July and are lowest in January — reflecting the dominance of Northern Hemisphere land mass in this dataset.

### 4.5 Land vs. Land+Ocean Temperature Comparison

This is where the ocean-impact angle comes in. The `Land + Ocean` line only starts around 1850 (see Section 2 — earlier ocean data doesn't exist).

In [14]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=yearly_land['dt'], y=yearly_land['LandAverageTemperature'],
                          mode='lines', name='Land Only', line=dict(color='orange')))

fig.add_trace(go.Scatter(x=yearly_land_and_ocean['dt'], y=yearly_land_and_ocean['LandAndOceanAverageTemperature'],
                          mode='lines', name='Land + Ocean', line=dict(color='blue')))

fig.update_layout(title='Land vs. Land+Ocean Temperature Comparison',
                   xaxis_title='Year', yaxis_title='Average Temperature (°C)')
fig.show()

## 5. Key Findings

- **Overall trend:** Global land temperature has risen significantly since 1750, with a sharp, steady increase visible from ~1975 onward — consistent with modern climate change.
- **Sharpest decadal rise (Land+Ocean):** The 2000s decade shows the highest combined land-ocean temperature on record.
- **Top warming countries:** Using decade-averaged comparisons, Uzbekistan and Turkmenistan show the highest warming, followed by China, Egypt, and the UAE — predominantly arid/continental climates.
- **Seasonal pattern:** Global temperatures peak in July (~14.3°C) and are lowest in January (~2.3°C).
- **Historical anomaly:** The temperature dip around 1810-1820 aligns with the 1815 Mount Tambora volcanic eruption, which caused global cooling.

## 6. Limitations & Notes

- The dataset's actual range is **1750-2015** (verified via `df['dt'].max()`), not 1750-2013 as the dataset description might suggest at first glance — always validate data ranges rather than trusting descriptions.
- Ocean temperature data is only available from ~1850 onward due to historical measurement limitations — these `NaN` values are left intentional, not imputed.
- Country-level warming uses decade-averaged comparisons (first 10 years vs. last 10 years) specifically to avoid the seasonal bias that a naive first-vs-last-row comparison would introduce.

---

*This notebook is part of a larger project — see the [GitHub repository](https://github.com/anshikaraj-hub/Global-Warming-And-Ocean-Impact-Analysis) for the full modular codebase, automated data download script, and project documentation.*